# Trajectories | Path Complexity
-----------------------------------------

## Setup and Configuration

In [ ]:
##### Imports ##########################################################################

#==== Generic Imports ================
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

#==== Movement Imports ===============
import movement.kinematics

#==== DataConduit Imports (Core) =====
from data_conduit.datastructures import slice_stream, slice_stream_per_trial
from data_conduit.integrations.DLC.pose import pose_to_movement     # Combine aligned pose and confidence in movement format.

#==== DataConduit Imports (Unique) ===
from movement_figures.data_template.loading import build_datastructure
from movement_figures.misc import visual_streams_check, configure_simplified_trial_df
from data_conduit.refactor_qc import (
    TRAINING_FIRST_MID_LAST_DETAIL,
    training_spec,
    filter_trials,
)

### Notebook Settings ######################################################

pd.set_option('display.max_columns', None)                         # Show all trial columns during inspection.
pd.set_option('display.max_rows', None)


In [ ]:
##### Setup Parameters #################################################################

# === 1| Select Mice, Days and Recording Folders Before Reading Files =============
ROOT = Path("/media/sepi/Elements/PathIntegrationProtocol/BonsaiOutput/Training")

LEVEL_SELECTORS = {
    'l0_selector': 'MR_M01569515', 
    'l1_selector': 'Day21'
    }                                                                                       # Example choices belong here: l0_selector and l1_selector.
# INCLUDE_SESSIONS = ('2026-06-21T093349Z',)                                                # Optional list of recording folder names; None keeps selected folders.
# STREAMS = ('trials', 'events', 'video', 'dlc')                                            # These figures require events/trials and aligned pose; other sources remain available.


#=== 2| Set the trial filtering parameters for the experiment protocol ============
spec = training_spec(TRAINING_FIRST_MID_LAST_DETAIL)                                        # Flattens `{training_detail}` table into per-session spec. 

# included_sessions = training_session_names(spec)                                          # Returns a list of session folder names that match the spec.
#   Not used here, as sessions have been filtered in another way. If other sessions 
#   not in the above spec are used, then applying the `filter_trials` function with 
#   the above spec could result in incorrect filtering.


In [ ]:
##### Load DataStructure ###############################################################

#=== Load data ========================================================================

fig_ds_obj = build_datastructure(
    root               = ROOT,                                                                 # Directory above the named hierarchy levels.
    level_names        = ("mouseID", "day"),                                                   # Lab layout: root / mouseID / day / session.
    streams            = ("trials", "session_settings", "video", "dlc"),                       # Data streams to be included in the datastructure.
    include            = None,                                                                 # Keep these session folder names; None keeps all.
    exclude            = None,                                                                 # Alternatively omit these session folder names.
    raw_events         = True,                                                                 # Convenience alias for adding "events" to the requested sources.
    device_yaml        = "./device.yml",                                                       # Lab Nosepoke board register definitions.
    soundcard_yaml     = "./soundcard.yml",                                                    # Lab SoundCard register definitions.
    nosepoke_count     = 18,                                                                   # Port count used by the existing Q_C parser.
    trial_start_buffer = 0.0,                                                                  # Preserve the existing lab parser setting.
    dlc_file_format     = "auto",                                                              # Existing reader prefers CSV, otherwise HDF5.
    trial_spec = None,                                                                         # Optional experiment rules; None keeps the existing Q_C specification. 
    #    Not used here, as sessions are already selected. 
    #    If session is not one in the spec, then subsequent use of 
    #   `filter_trials` to apply correct experiment protocol rules could be incorrect.
    
    **LEVEL_SELECTORS,
)


#=== Select and load the streams of interest from datastructure. ======================
fig_ds_obj.select()
streams = fig_ds_obj.load()                                                                    # Load is required to access the selected streams.

#=== Create convenient compact trial dataframe without some unnecessary columns. ======
streams['compact_trials'] = configure_simplified_trial_df(              
    trial_df = streams['trials'],
    modify_in_place = False,                                                                   # Whether to modify the trial dataframe in place or return a new dataframe. 
)
'''More useful for visual inspection and analysis. 
Check the misc.py file for the default columns to keep and drop.'''


#=== Apply Trial Filtering Protocol to construct new stream objects ===================

# Full filtered trials
streams['filtered_trials'] = filter_trials(
    trials = streams['trials'],
    spec = spec,
    # session_column = 'session', mouse_column = 'mouseID', trial_index_column = 'trial_index', led_column = 'LED', report = True,
    #                                                            # Leave params untouched for now, let the function use its defaults.
)

# Filtered compact trials 
streams['filtered_compact_trials'] = filter_trials(
    trials = streams['compact_trials'],
    spec = spec,
    # session_column = 'session', mouse_column = 'mouseID', trial_index_column = 'trial_index', led_column = 'LED', report = True,
    #                                                            # Leave params untouched for now, let the function use its defaults.
)

#=== Merge Pose & Confidence Streams ==================================================
streams['dlc:movement'] = pose_to_movement(
    slice_stream(streams["dlc:position"],   
                #selectors={"session": session_id}
    ),
    slice_stream(streams["dlc:confidence"], 
                 #selectors={"session": session_id}
    ),
)

#=== Visual streams check =============================================================
visual_streams_check(
    loaded_ds_object = streams,                                                                # Loaded dataset object containing all streams of interest.
    show_stream_shapes = True,                                                                 # Whether to display the shapes of the streams.
    show_stream_dtypes = True,                                                                 # Whether to display the data types of the streams.
    display_streams = False                                                                    # Whether to display the actual stream data as a table.
)



## Select Trials to Show

In [ ]:
##### Trial Selection ##################################################################

#==== Extract Trial Range ============
selected_trials = slice_stream(streams['filtered_trials'][:8])              # Select the first n trials

#==== Extract Recording Identifier ===
sessionID = selected_trials['session'].unique().item()                      # Extract unique sID from selected trials, remove duplicates, ensure single value.

#==== Select Tracked Positions =======
recording = slice_stream(stream = streams['dlc:movement'],                  # selectors{} here keeps movement data belonging to the specified session only.
                        selectors = {'session' : sessionID }, 
)

position = recording['position']                                            # ['position'] extracts positional data from the movement stream.

position = position.sel(individual='individual_0',                          # .sel() selects the specified individual from the positional data.
                        drop=True,                                          # drop=True removes the individual dimension from the resulting DataArray, but leaves other dims intact.
)


## Select Outbound and Inbound Positions

In [ ]:
##### Prepare Trial Paths for the Complexity Calculation ###############################

#==== Select the Measurement and Units ===
COMPLEXITY_MEASURE = 'length'                                     # 'length', 'sinuosity', or 'emax'.
EMAX_IN_SPATIAL_UNITS = True                                      # False selects dimensionless Emax.
SPATIAL_UNIT = 'Pixels'
PIXEL_UNIT_CONV_METRIC = None                                     # Output distance units per pixel; None keeps pixels.


#==== Convert Position Before Measuring ==
body_position = position.sel(keypoint='body', drop=True)
if PIXEL_UNIT_CONV_METRIC is not None:
    body_position = body_position * PIXEL_UNIT_CONV_METRIC        # Scale positions before calculating all path measures.


#==== Slice Each Trial's Outbound Path ====
outbound = slice_stream_per_trial(
    stream     = body_position,
    trials     = selected_trials,
    segment    = 'outbound',
    time_coord = 'time',
)


#==== Slice Each Trial's Inbound Path =====
inbound = slice_stream_per_trial(
    stream     = body_position,
    trials     = selected_trials,
    segment    = 'inbound',
    time_coord = 'time',
)


## Compute the Selected Complexity Measure

The selectable outbound measure is **path length**, **sinuosity**, or **Emax**. The return measure remains the mean perpendicular distance to the infinite line through the observed inbound endpoints, as in `trajectories_fixed`; it is not error relative to the true home port.

The guide's complete-path rule is retained: an incomplete phase is left as NaN rather than filled or shortened. This is a measurement choice, not a new data-validation layer.

References: [length](https://movement.neuroinformatics.dev/latest/api/movement.kinematics.compute_path_length.html), [sinuosity](https://movement.neuroinformatics.dev/dev/api/movement.kinematics.compute_path_sinuosity.html), [Emax](https://movement.neuroinformatics.dev/dev/api/movement.kinematics.compute_path_emax.html), [deviation](https://movement.neuroinformatics.dev/dev/api/movement.kinematics.compute_path_deviation.html).

In [ ]:
##### Compute Path Complexity Using `movement` #########################################


def compute_path_metrics(
    outbound: list[xr.DataArray],
    inbound: list[xr.DataArray],
    selected_trials: pd.DataFrame,
    measure: str = 'length',
    emax_in_spatial_units: bool = True,
) -> pd.DataFrame:
    """Pair one outbound measure with mean inbound deviation for each trial.

    Parameters
    ----------
    outbound, inbound : list[xarray.DataArray]
        One path per trial, in selected_trials row order. Convert units beforehand.
    selected_trials : pandas.DataFrame
        Trial rows retained in the returned metric table.
    measure : str
        Outbound measure: 'length', 'sinuosity', or 'emax'.
    emax_in_spatial_units : bool
        For Emax only: True returns position units; False is dimensionless.

    Returns
    -------
    pandas.DataFrame
        Trial columns plus outbound_<measure> and inbound_mean_deviation.
        Incomplete phases remain NaN; infinite Emax remains in the table.
    """
    #==== 1| Set the Number of Positions Needed by the Measure ========
    if measure == 'length':
        minimum_points = 2                                      # One displacement is sufficient for path length.
    elif measure in ('sinuosity', 'emax'):
        minimum_points = 3                                      # A turning angle requires two successive steps.
    else:
        raise ValueError("measure must be 'length', 'sinuosity', or 'emax'.")

    outbound_values = []
    inbound_values = []


    #==== 2| Measure Each Trial Without Joining Its Neighbours ========
    for outward, returning in zip(outbound, inbound, strict=True):
        outbound_value = np.nan
        inbound_value = np.nan

        # Preserve the guide's rule: do not measure a phase with missing positions.
        if outward.sizes['time'] >= minimum_points and bool(np.isfinite(outward).all()):
            if measure == 'length':
                outbound_value = float(movement.kinematics.compute_path_length(outward))
            elif measure == 'sinuosity':
                outbound_value = float(movement.kinematics.compute_path_sinuosity(outward))
            else:
                outbound_value = float(movement.kinematics.compute_path_emax(
                    outward,
                    in_spatial_units=emax_in_spatial_units,
                ))

        # The observed inbound endpoints must define a nonzero reference line.
        if returning.sizes['time'] >= 2 and bool(np.isfinite(returning).all()):
            if bool((returning.isel(time=0) != returning.isel(time=-1)).any()):
                deviation = movement.kinematics.compute_path_deviation(returning)
                inbound_value = float(deviation.mean('time', skipna=False))

        outbound_values.append(outbound_value)
        inbound_values.append(inbound_value)


    #==== 3| Attach the Measurements to the Original Trial Rows =======
    metrics = selected_trials.copy()
    metrics[f'outbound_{measure}'] = outbound_values
    metrics['inbound_mean_deviation'] = inbound_values

    return metrics


In [ ]:
##### Calculate the Selected Path Measure ##############################################

metrics = compute_path_metrics(
    outbound              = outbound,
    inbound               = inbound,
    selected_trials       = selected_trials,
    measure               = COMPLEXITY_MEASURE,
    emax_in_spatial_units = EMAX_IN_SPATIAL_UNITS,
)

display(metrics[['trial_index', 'outcome', f'outbound_{COMPLEXITY_MEASURE}', 'inbound_mean_deviation']])


## Path Complexity Plotting Function

In [ ]:
##### Plot Path Complexity on Ax #######################################################


def plot_ax_path_complexity(
    ax: plt.Axes,
    metrics: pd.DataFrame,
    measure: str = 'length',
    spatial_unit: str = 'Pixels',
    emax_in_spatial_units: bool = True,
    group_by: str | None = 'outcome',
    subtitle: str | None = 'Outbound Complexity versus Inbound Deviation',
    **kwargs,
) -> plt.Axes:
    """Plot the selected outbound measure against the matching return deviation.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axis on which to draw the scatter plot.
    metrics : pandas.DataFrame
        Already-calculated trial measurements from compute_path_metrics.
    measure : str
        'length', 'sinuosity', or 'emax'; must match the calculated column.
    spatial_unit : str
        Position units used when computing metrics; labels only, not conversion.
    emax_in_spatial_units : bool
        Must match the calculation so Emax is labelled with the correct units.
    group_by : str or None
        Trial column defining scatter groups. None draws a single group.
    subtitle : str or None
        Plot title. None omits it.
    **kwargs
        Arguments passed to Axes.scatter, such as s and alpha.

    Returns
    -------
    matplotlib.axes.Axes
        Supplied axis. Nonfinite pairs are not drawn or removed from metrics.
    """
    #==== 1| Label the Selected Measure and Its Actual Units ==========
    if measure == 'length':
        xlabel = f'Outbound Path Length ({spatial_unit})'
    elif measure == 'sinuosity':
        xlabel = f'Outbound Sinuosity (1 / sqrt({spatial_unit}))'
    elif measure == 'emax':
        emax_unit = spatial_unit if emax_in_spatial_units else 'Dimensionless'
        xlabel = f'Outbound Emax ({emax_unit})'
    else:
        raise ValueError("measure must be 'length', 'sinuosity', or 'emax'.")


    #==== 2| Plot the Already-Paired Measurements ======================
    if group_by is None:
        ax.scatter(metrics[f'outbound_{measure}'], metrics['inbound_mean_deviation'], **kwargs)
    else:
        for label, group in metrics.groupby(group_by, sort=False):
            ax.scatter(group[f'outbound_{measure}'], group['inbound_mean_deviation'],
                       label=label, **kwargs)
        ax.legend(title=group_by)


    #==== 3| Add Measurement Labels and the Plot Title ================
    ax.set(xlabel=xlabel, ylabel=f'Mean Inbound Deviation ({spatial_unit})')
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.2)
    ax.set_axisbelow(True)

    if subtitle is not None:
        ax.set_title(subtitle)

    return ax


## Generate Plot

In [ ]:
##### Plot the Selected Complexity Measure #############################################

fig, ax = plt.subplots(figsize=(7, 5), layout='constrained')

ax = plot_ax_path_complexity(
    ax                    = ax,
    metrics               = metrics,
    measure               = COMPLEXITY_MEASURE,
    spatial_unit          = SPATIAL_UNIT,
    emax_in_spatial_units = EMAX_IN_SPATIAL_UNITS,
    group_by              = 'outcome',
    subtitle              = f'Outbound {COMPLEXITY_MEASURE} versus Inbound Deviation',
    s                     = 38,
    alpha                 = 0.8,
)

fig.savefig(f'path_complexity_{COMPLEXITY_MEASURE}.svg', bbox_inches='tight')
metrics.to_csv(f'path_metrics_{COMPLEXITY_MEASURE}.csv', index=False)
plt.show()


Change `COMPLEXITY_MEASURE` and rerun the calculation and plot cells. Length and spatial Emax use position units; sinuosity uses inverse square-root position units. Higher Emax indicates greater directional persistence, not greater tortuosity. Infinite Emax is retained in the table but cannot appear as a finite scatter point.

The new complexity calls require a movement build exposing `compute_path_sinuosity` and `compute_path_emax`. No custom substitute is used when a function is unavailable.